In [1]:
import numpy as np
import pandas as pd
import pandapower as pp
import simbench as sb
from pathlib import Path

RANDOM_SEED = 42
SAMPLES_PER_SCENARIO = 25
SIMBENCH_CODE = "1-MVLV-urban-all-0-sw"

rng = np.random.default_rng(RANDOM_SEED)

In [2]:
grid = sb.get_simbench_net(SIMBENCH_CODE)

load_p = sb.get_absolute_profiles_from_relative_profiles(
    grid, element="load", multiplying_column="p_mw",
    profile_suffix="_pload", time_as_index=True,
)

load_q = sb.get_absolute_profiles_from_relative_profiles(
    grid, element="load", multiplying_column="q_mvar",
    profile_suffix="_qload", time_as_index=True,
)

renew_p = sb.get_absolute_profiles_from_relative_profiles(
    grid, element="sgen", multiplying_column="p_mw",
    time_as_index=True,
)

print("Grid buses:", len(grid.bus))
print("Load profile shape:", load_p.shape)
print("Reactive profile shape:", load_q.shape)
print("Renewable profile shape:", renew_p.shape)

Grid buses: 10458
Load profile shape: (35136, 11542)
Reactive profile shape: (35136, 11542)
Renewable profile shape: (35136, 806)


Define the Scenarios

In [3]:
SCENARIOS = {
    "Normal":          ((0.9, 1.1), (0.9, 1.1), (0.00, 0.00), "none", None),
    "High Demand":     ((1.3, 1.6), (0.8, 1.1), (0.00, 0.00), "none", None),
    "Low Renewable":   ((0.9, 1.2), (0.2, 0.5), (0.00, 0.00), "none", None),
    "High EV":         ((1.0, 1.2), (0.8, 1.0), (0.15, 0.25), "none", None),
    "Battery Charging":((0.8, 1.0), (1.0, 1.2), (0.00, 0.00), "charge", None),
    "Battery Support": ((1.2, 1.5), (0.4, 0.7), (0.00, 0.00), "support", None),
    "Line Outage":     ((0.9, 1.1), (0.8, 1.1), (0.00, 0.00), "none", 10229),
    "Combined Stress": ((1.4, 1.8), (0.2, 0.5), (0.15, 0.25), "none", None),
    "Extreme Stress":  ((2.5, 4.0), (0.1, 0.3), (0.25, 0.40), "none", 10229),
}

Generate 225 Scenario Samples

In [7]:
rows = []

for name, config in SCENARIOS.items():
    load_range, renew_range, ev_range, battery_mode, outage = config

    chosen_steps = rng.choice(
        len(load_p), SAMPLES_PER_SCENARIO, replace=False
    )

    for step in chosen_steps:
        rows.append({
            "scenario": name,
            "profile_step": int(step),
            "timestamp": load_p.index[step],
            "load_scale": rng.uniform(*load_range),
            "renewable_scale": rng.uniform(*renew_range),
            "ev_penetration": rng.uniform(*ev_range),
            "battery_mode": battery_mode,
            "outaged_line": outage,
        })

Create DataFrame

In [10]:
scenario_df = pd.DataFrame(rows)

scenario_df["sample_id"] = np.arange(1, len(scenario_df) + 1)
scenario_df["outaged_line"] = scenario_df["outaged_line"].astype("Int64")

scenario_df = scenario_df[
    ["sample_id", "scenario", "profile_step", "timestamp",
     "load_scale", "renewable_scale",
     "ev_penetration", "battery_mode", "outaged_line"]
]

display(scenario_df.head())

,sample_id,scenario,profile_step,timestamp,load_scale,renewable_scale,ev_penetration,battery_mode,outaged_line
0,1,Normal,22924,26.08.2016 20:00,0.919396,1.069159,0.0,none,<NA>
1,2,Normal,16451,20.06.2016 09:45,1.008473,0.933062,0.0,none,<NA>
2,3,Normal,18333,10.07.2016 00:15,0.932062,1.070387,0.0,none,<NA>
3,4,Normal,3156,02.02.2016 21:00,1.047026,0.959207,0.0,none,<NA>
4,5,Normal,7096,14.03.2016 22:00,0.980971,1.052003,0.0,none,<NA>


Add Base Grid Values

In [11]:
base_p = load_p.sum(axis=1).to_numpy()
base_q = load_q.sum(axis=1).to_numpy()
base_renew = renew_p.sum(axis=1).to_numpy()

steps = scenario_df["profile_step"].to_numpy()

scenario_df["base_load_mw"] = base_p[steps]
scenario_df["base_q_mvar"] = base_q[steps]
scenario_df["base_renewable_mw"] = base_renew[steps]

scenario_df["scaled_load_mw"] = (
    scenario_df["base_load_mw"] * scenario_df["load_scale"]
)

Validate Everything

In [12]:
assert len(scenario_df) == 225
assert scenario_df["scenario"].nunique() == 9
assert scenario_df["timestamp"].notna().all()
assert scenario_df["load_scale"].gt(0).all()
assert scenario_df["renewable_scale"].gt(0).all()
assert scenario_df["ev_penetration"].between(0, 1).all()

print("Scenario generation successful.")
print("Scenarios:", scenario_df["scenario"].nunique())
print("Samples per scenario:", SAMPLES_PER_SCENARIO)
print("Total samples:", len(scenario_df))

Scenario generation successful.
Scenarios: 9
Samples per scenario: 25
Total samples: 225


Check Distribution

In [13]:
summary = scenario_df.groupby("scenario").agg(
    samples=("sample_id", "count"),
    avg_load_scale=("load_scale", "mean"),
    avg_renew_scale=("renewable_scale", "mean"),
    avg_ev_penetration=("ev_penetration", "mean"),
)

display(summary.round(3))

,samples,avg_load_scale,avg_renew_scale,avg_ev_penetration
scenario,,,,
Battery Charging,25,0.902,1.092,0.000
Battery Support,25,1.331,0.561,0.000
Combined Stress,25,1.562,0.382,0.204
Extreme Stress,25,3.270,0.188,0.334
High Demand,25,1.469,0.957,0.000
High EV,25,1.118,0.878,0.191
Line Outage,25,1.006,0.947,0.000
Low Renewable,25,1.054,0.355,0.000
Normal,25,1.000,0.995,0.000


Export

In [14]:
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

OUTPUT_PATH = DATA_DIR / "scenario_catalog.csv"

scenario_df.to_csv(OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH)
print("Rows:", len(scenario_df))

Saved: data\scenario_catalog.csv
Rows: 225
